In [2]:
import pandas as pd
import re 
import numpy as np
import nltk
from nltk.tokenize import word_tokenize
from nltk import pos_tag, ne_chunk

In [3]:
data = pd.read_csv("results---From---2023-10-31--22-08-07---To---2024-04-07--16-02-28.csv")
data = data[['message']].dropna()
data = data.drop_duplicates()
data

,message
0,#Renta x DÍAS de Apto en el Vedado
1,No tienes permisos para ejecutar este comando ...
2,/revisarbrplus@ReputacionPlusBot
8,Casa en venta en la zona sur cerca de las fábr...
9,Busco renta por tiempo indefinido para una par...
...,...
4988,Busco alquiler en el vedado límite 150 verde s...
4992,"Busco alquiler por tiempo indefinido, 58316712"
4993,Busco alquiler en la lisa o lo más cerca posible
4994,Busco alquiler en la Lisa


### Limpiando los datos para poder empezar a trabajar sobre ellos

In [5]:
def delete_emojis(text):
    patron_emojis = re.compile(pattern="["
                                      u"\U0001F600-\U0001F64F"  
                                      u"\U0001F300-\U0001F5FF"  
                                      u"\U0001F680-\U0001F6FF"  
                                      u"\U0001F700-\U0001F77F"  
                                      u"\U0001F780-\U0001F7FF"  
                                      u"\U0001F800-\U0001F8FF"  
                                      u"\U0001F900-\U0001F9FF"  
                                      u"\U0001FA00-\U0001FAFF" 
                                      u"\U00002702-\U000027B0"  
                                      u"\U00002702-\U000027B0"
                                      u"\U000024C2-\U0001F251"
                                      "]+", flags=re.UNICODE)
    return patron_emojis.sub(r'', text)

def delete_commands(texto):
    cleaned_text = re.sub(r'/\S+', '', texto)
    cleaned_text = re.sub(r'@\S+', '', cleaned_text)
    cleaned_text = re.sub(r'#', '', cleaned_text)
    cleaned_text = re.sub(r'http[s]?://\S+', '', cleaned_text)
    cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()
    
    return cleaned_text

def tokenize(text):
    return word_tokenize(text)

def normalize_text(tokens):
    return [token.lower() for token in tokens]

def pos_tagging_spacy(text):
    doc = nlp(text)
    return [(token.text, token.pos_) for token in doc]

def lemmatize(text):
    doc = nlp(text)
    lemmatized_text = ' '.join([token.lemma_ for token in doc])
    
    return lemmatized_text

def extract_entities(text):
    doc = nlp(text)
    return [(ent.text, ent.label_) for ent in doc.ents]


In [6]:
data['message'] = data['message'].apply(delete_emojis)
data['message'] = data['message'].apply(delete_commands)
data = data.drop_duplicates()
data

,message
0,Renta x DÍAS de Apto en el Vedado
1,No tienes permisos para ejecutar este comando ...
2,
8,Casa en venta en la zona sur cerca de las fábr...
9,Busco renta por tiempo indefinido para una par...
...,...
4988,Busco alquiler en el vedado límite 150 verde s...
4992,"Busco alquiler por tiempo indefinido, 58316712"
4993,Busco alquiler en la lisa o lo más cerca posible
4994,Busco alquiler en la Lisa


#### Ahora vamos a empezar a extraer features de interés y vamos empezar con el precio y la moneda en que se haria la negociación

In [ ]:
def extract_price(message):
    prices = re.findall(r'\b\d{2,6}(?:[.,]\d+)? | \d{2,6}(?:[.,]\d+)? mil\b', message)
    if prices:
        return prices
    return None


def extract_currency(message):
    currencies = re.findall(r'\b(USD|usd|dólar|dolar|EURO|euro|MLC|mlc|CUP|cup|pesos|mn|dolar|dolares|mil)\b', message, re.IGNORECASE)
    if currencies:
        return currencies
    return None

In [ ]:
def extract_location(message):
    doc = nlp(message)
    location = [ent.text for ent in doc.ents if ent.label_ == 'LOC']
    return location

data= data.dropna()
print(data)